# CLING tutorial

**CLING** (Cross-view Latent Integration via Nonparametric Gamma Shrinkage) is an
unsupervised Bayesian multi-view factor model for multi-omics integration. This
notebook is a complete, self-contained walkthrough on a small **synthetic**
dataset - no downloads, no credentials, and it runs top-to-bottom in a few
seconds on a laptop.

You will:

1. build correctly shaped multi-view data (including missing values),
2. fit the primary CLING model with a fixed seed,
3. inspect ELBO convergence,
4. extract factors and loadings,
5. read the variance-explained diagnostics and the effective number of factors,
6. save and reload a fit, and
7. run the `CLING-MGP` and `CLING-ARD` ablation variants.

**Prerequisites:** from the repository root, `pip install ".[tutorial]"`
(installs CLING plus `matplotlib`).
**Approximate runtime:** under ~10 seconds total.

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import cling

print("cling version:", cling.__version__)
rng = np.random.default_rng(0)   # a fixed generator for reproducible data

## 2. Build a multi-view dataset

CLING takes `M` views measured on the same `N` samples. View `m` is an
`N x D_m` matrix (samples x features); `D_m` may differ across views. Here we
draw a shared latent `Z` of true rank 5 and generate three views as noisy linear
mixtures, so we know the ground-truth number of factors.

In [ ]:
N = 120                       # samples (shared across views)
true_K = 5                    # ground-truth number of latent factors
dims = [40, 30, 20]           # features per view (may differ)

Z_true = rng.standard_normal((N, true_K))
views = []
for D in dims:
    W = rng.standard_normal((D, true_K))
    views.append(Z_true @ W.T + 0.25 * rng.standard_normal((N, D)))

print("N =", N)
for m, v in enumerate(views):
    print(f"view {m}: shape {v.shape}")   # expect (120, 40), (120, 30), (120, 20)

## 3. Missing values

`NaN` entries are treated as missing and handled directly under a missingness
mask - CLING does **not** require imputation. We knock out ~8% of the entries in
view 0 to demonstrate this.

In [ ]:
missing = rng.random(views[0].shape) < 0.08
views[0][missing] = np.nan
print(f"view 0 now has {int(np.isnan(views[0]).sum())} missing entries "
      f"({100 * np.isnan(views[0]).mean():.1f}%)")

## 4. Preprocessing

CLING's required preprocessing is per-feature centering, which is applied by
default (`center=True`). The `MultiviewDataset` container performs it and reports
basic dataset properties; `cling.fit` builds it for you, so you rarely construct
it directly.

In [ ]:
dataset = cling.MultiviewDataset.from_arrays(views, view_names=["rna", "atac", "met"])
print("views (M):", dataset.M, "| samples (N):", dataset.N, "| features (D):", dataset.D)

## 5. Fit the primary CLING model

We fit with a fixed, overcomplete truncation ceiling `K_init` (here 20; the paper
uses 30) and a reproducible `seed`. The shrinkage hierarchy switches off
unsupported factors, so the number of *active* factors is learned automatically.
For `N < 1000`, `cling.fit` applies the mild `Gamma(3, 2.5)` shrinkage default.

In [ ]:
fitted = cling.fit(
    views,
    K_init=20,
    seed=0,
    max_iter=600,
    view_names=["rna", "atac", "met"],
)
print("iterations run:", fitted.training.n_iterations)
print("final ELBO:", round(fitted.training.final_elbo, 2))

## 6. ELBO convergence

CAVI maximises the evidence lower bound (ELBO), which increases monotonically and
then plateaus. Plotting the trace is the standard convergence check; we also
verify the trace is monotone and that its relative change has become tiny. (At
the default strict `"slow"` tolerance an overcomplete fit may run to the
iteration cap while the ELBO has effectively plateaued, which is expected.)

In [ ]:
elbo = np.asarray(fitted.training.elbo_history)
plt.figure(figsize=(5, 3))
plt.plot(elbo)
plt.xlabel("iteration"); plt.ylabel("ELBO"); plt.title("ELBO convergence")
plt.tight_layout(); plt.show()

diffs = np.diff(elbo)
print("ELBO increased monotonically:",
      bool(np.all(diffs >= -1e-6 * np.abs(elbo[:-1]) - 1e-8)))
tail = np.abs(np.diff(elbo[-50:])) / (np.abs(elbo[-50:-1]) + 1e-12)
print("max relative ELBO change over last 50 iterations:", f"{tail.max():.2e}")

## 7. Factors and loadings

`get_factors()` returns the shared latent scores `Z` of shape `(N, K)`;
`get_weights()` returns one loading matrix `W^m` of shape `(D_m, K)` per view.

In [ ]:
Z = fitted.get_factors()
W = fitted.get_weights()
print("Z (factors):", Z.shape)                      # (N, K)
for m, Wm in enumerate(W):
    print(f"W[{m}] (loadings):", Wm.shape)           # (D_m, K)

## 8. Variance-explained diagnostics and the effective number of factors

A factor is *active* when its per-view explained variance `R^2` reaches the
threshold `epsilon = 0.01` in at least one view. The count of active factors is
`fitted.K`, and it should recover the true rank (5) of our synthetic data.

In [ ]:
r2_view = fitted.variance_explained_per_view()
r2_factor = fitted.variance_explained_per_factor()

print("per-view R^2:", np.round(r2_view, 3))
print("effective number of active factors (fitted.K):", fitted.K)
print("factors with R^2 >= 0.01:", int((r2_factor >= 0.01).sum()), "(true rank = 5)")

order = np.argsort(r2_factor)[::-1]
plt.figure(figsize=(5, 3))
plt.bar(range(len(r2_factor)), r2_factor[order])
plt.axhline(0.01, color="red", ls="--", lw=1, label="epsilon = 0.01")
plt.xlabel("factor (sorted)"); plt.ylabel("R^2"); plt.legend()
plt.title("Per-factor variance explained"); plt.tight_layout(); plt.show()

## 9. Interpreting the output dimensions

`Z[n, k]` is sample `n`'s score on factor `k`; `W[m][d, k]` is the loading of
feature `d` in view `m` on factor `k`. The reconstruction of view `m` is
`Z @ W[m].T`. Factors are shared across views, while each view has its own
loadings - this is what lets a single factor express coordinated signal across
modalities.

In [ ]:
recon0 = fitted.reconstruct(view=0)
print("reconstruction of view 0:", recon0.shape)     # (N, D_0)

## 10. Saving and loading a fit

A fit is saved to a compressed `.npz` archive (factors, loadings, diagnostics and
metadata) and reloaded as a read-only snapshot.

In [ ]:
import tempfile, os
path = os.path.join(tempfile.mkdtemp(), "cling_fit.npz")
fitted.save(path)

reloaded = cling.FittedModel.load(path)
print("round-trip factors identical:",
      bool(np.array_equal(fitted.get_factors(), reloaded.get_factors())))
print("stored variant:", reloaded.model.variant)

## 11. Ablation variants

The paper's two ablations are exposed under the display names `CLING-MGP` (a
single local `Gamma` prior instead of the Gamma-Gamma hierarchy) and `CLING-ARD`
(a view-level precision with cumulative shrinkage across factors only). They use
the same API.

In [ ]:
for variant in ["CLING", "CLING-MGP", "CLING-ARD"]:
    f = cling.fit(views, K_init=20, seed=0, max_iter=200, variant=variant)
    k_eff = int((f.variance_explained_per_factor() >= 0.01).sum())
    print(f"{variant:>10}: active factors = {k_eff}")

## Summary

You built multi-view data with missing values, fit CLING, confirmed the ELBO
rose monotonically and plateaued, extracted factors and loadings, read the variance-explained
diagnostics, recovered the true number of factors automatically, saved and
reloaded a fit, and ran the two ablation variants. For the paper's operating
point on real data (`K_init = 30`, sample-size-aware shrinkage, 25 seeds), see
the `reproducibility/` directory.